# Notebook 10: Class Imbalance & Threshold Tuning

**Difficulty:** ⭐⭐⭐⭐ | **Estimated time:** 2.5 hours | **Week 6 — Classification: Logistic Regression & Decision Boundaries**

---

## Why Does This Matter?

Real-world classification is almost never balanced. Spam is ~5% of email. Cancer is ~0.1% of screenings. Fraud is ~0.01% of transactions. A model that labels every single email as "not spam" achieves **95% accuracy** without learning anything — and is completely useless.

This notebook addresses two complementary strategies:

1. **Threshold tuning (post-training):** The model already outputs probabilities. You control where you draw the line between "spam" and "not spam."
2. **Weighted loss (during training):** Tell the optimiser that mistakes on the minority class are more costly, so it tries harder to get them right.

> *This notebook is a companion to Week 4 Notebook 07 (SMOTE / oversampling). That notebook rebalances the training data. This notebook keeps the data as-is but adjusts the model's decision logic.*

**Spam detection context:** You are the ML engineer at an email provider. 5% of incoming emails are spam. Your task: build a filter that is actually useful — not one that ignores the problem by predicting "not spam" every time.

## Real-World Analogy: The Airport Security Scanner

Imagine an airport security scanner set to detect weapons. Threats are rare — perhaps 1 in 10,000 bags.

- **If the threshold is too low (flag everything):** Every second passenger gets a pat-down. Security is exhausted, passengers miss flights, and real threats are buried in the noise. High recall, terrible precision.
- **If the threshold is too high (flag nothing):** Nobody is inconvenienced, but actual weapons pass through. Perfect precision (no false alarms), disastrous recall (threats missed).
- **The right threshold** depends on the **cost** of each error type: missing a weapon vs. inconveniencing a passenger. That cost ratio is a business/ethical decision — the machine learning engineer's job is to implement whatever threshold follows from that decision.

| Security concept | Spam filter concept |
|---|---|
| Weapon in bag | Spam email |
| Legitimate bag flagged | Legitimate email marked spam (FP) |
| Weapon bag passes through | Spam gets through (FN) |
| Sensitivity dial on scanner | Classification threshold |
| Cost of missed weapon | Cost of FN |
| Cost of false alarm | Cost of FP |

## Plain-English Concepts Before the Code

### The Accuracy Paradox
On a dataset with 95% class-0 examples, a classifier that predicts class 0 for everything achieves **95% accuracy**. This is the **accuracy paradox** — accuracy is a misleading metric when classes are imbalanced.

### Precision and Recall (for the spam / minority class)
$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of emails you flagged, how many were actually spam?)}$$
$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of all real spam, how many did you catch?)}$$

### F-beta Score: Tuning the Precision-Recall Tradeoff
$$F_\beta = (1+\beta^2) \cdot \frac{P \cdot R}{\beta^2 \cdot P + R}$$

| β value | Meaning | When to use |
|---|---|---|
| 0.5 | Precision matters 2× more than recall | Spam filter (false alarms annoy users) |
| 1.0 | Precision and recall equally weighted | General purpose |
| 2.0 | Recall matters 2× more than precision | Cancer screening (missing a case is catastrophic) |

### Youden's J Statistic (for medical / balanced-cost scenarios)
$$J = \text{TPR} - \text{FPR} = \text{Sensitivity} + \text{Specificity} - 1$$
The threshold that maximises J is the one that simultaneously maximises recall and minimises false alarms.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, auc,
    average_precision_score, roc_auc_score,
    f1_score, fbeta_score, precision_score, recall_score
)

SEED = 42
np.random.seed(SEED)

# ── Consistent colour scheme ──────────────────────────────────────────────────
SPAM_COLOR    = '#d62728'   # red  — spam / class 1
NOSPAM_COLOR  = '#1f77b4'   # blue — not spam / class 0
OPTIMAL_COLOR = '#2ca02c'   # green — optimal threshold marker

print("Libraries loaded.")

## Section 1 — Creating a Severely Imbalanced Spam Dataset

We simulate an email dataset with realistic imbalance: **5% spam, 95% not-spam**.

**Features (2D for visualisability):**
- Feature 0: `exclamation_ratio` — fraction of characters that are `!`
- Feature 1: `suspicious_link_count` — number of suspicious URLs in the email

Spam emails have higher values on both features; legitimate emails have lower values.

In [ ]:
# ── Generate imbalanced spam dataset ─────────────────────────────────────────
rng = np.random.default_rng(SEED)

N_TOTAL  = 10_000   # total emails
N_SPAM   =    500   # 5% spam  → severe imbalance
N_LEGIT  =  9_500   # 95% legitimate

# Legitimate emails: low exclamation ratio, few suspicious links
X_legit = rng.multivariate_normal(
    mean=[0.5, 1.0],
    cov=[[0.5, 0.1], [0.1, 0.8]],
    size=N_LEGIT
)

# Spam emails: higher exclamation ratio, more suspicious links
X_spam = rng.multivariate_normal(
    mean=[2.0, 3.5],
    cov=[[0.8, 0.3], [0.3, 1.2]],
    size=N_SPAM
)

# Combine and create labels (0=legit, 1=spam)
X = np.vstack([X_legit, X_spam])
y = np.hstack([np.zeros(N_LEGIT), np.ones(N_SPAM)])

# Shuffle so the ordering is random (not all 0s then all 1s)
shuffle_idx = rng.permutation(len(y))
X, y = X[shuffle_idx], y[shuffle_idx]

# Train / test split — preserve the imbalance ratio with stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)

print(f"Total samples : {len(y):,}")
print(f"Train size    : {len(y_train):,}  (spam: {y_train.sum():.0f}, "
      f"legit: {(y_train==0).sum():.0f})")
print(f"Test size     : {len(y_test):,}   (spam: {y_test.sum():.0f}, "
      f"legit: {(y_test==0).sum():.0f})")
print(f"Imbalance ratio: 1 spam per {N_LEGIT/N_SPAM:.0f} legit emails")

## Section 2 — The Accuracy Paradox

Before training anything, let's prove that a "do-nothing" classifier can score 95% accuracy. This is why accuracy is the wrong metric for imbalanced data.

In [ ]:
# ── Demonstrate the accuracy paradox ─────────────────────────────────────────
# Baseline strategy: predict 'not spam' (class 0) for every single email
y_pred_baseline = np.zeros_like(y_test)  # all zeros = all 'not spam'

baseline_accuracy = (y_pred_baseline == y_test).mean()
baseline_spam_recall = 0.0   # we never predict spam, so we catch 0 spam
baseline_spam_precision = 0.0  # technically undefined (no positive predictions)

print("━" * 55)
print(" BASELINE: Always predict 'Not Spam'")
print("━" * 55)
print(f"  Accuracy          : {baseline_accuracy:.2%}  ← looks great!")
print(f"  Spam recall       : {baseline_spam_recall:.2%}  ← catastrophic")
print(f"  Spam precision    : N/A (no spam predicted)")
print(f"  Spam caught       : 0 out of {int(y_test.sum())}")
print("")
print("LESSON: 95% accuracy means the classifier learned NOTHING.")
print("It is simply exploiting the class imbalance.")
print("━" * 55)

# Visualise class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of class counts
counts = pd.Series(y_test).value_counts().sort_index()
axes[0].bar(['Not Spam (0)', 'Spam (1)'], counts.values,
             color=[NOSPAM_COLOR, SPAM_COLOR], edgecolor='black', linewidth=0.8)
axes[0].set_title('Test Set Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, f'{v:,}\n({v/len(y_test):.1%})',
                  ha='center', fontweight='bold')

# Raw scatter of the test data
mask0 = y_test == 0
mask1 = y_test == 1
axes[1].scatter(X_test[mask0, 0], X_test[mask0, 1],
                 c=NOSPAM_COLOR, s=5, alpha=0.3, label='Not Spam')
axes[1].scatter(X_test[mask1, 0], X_test[mask1, 1],
                 c=SPAM_COLOR, s=20, alpha=0.7, label='Spam')
axes[1].set_title('Test Feature Space', fontweight='bold')
axes[1].set_xlabel('Exclamation Ratio')
axes[1].set_ylabel('Suspicious Link Count')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 3 — Training the Baseline Model (Threshold = 0.5)

We train logistic regression with default settings and observe what happens at the default 0.5 threshold.

In [ ]:
# ── Train logistic regression with default settings ───────────────────────────
lr_default = Pipeline([
    ("scaler", StandardScaler()),
    # class_weight=None means all samples are weighted equally (default behaviour)
    ("lr", LogisticRegression(class_weight=None, max_iter=1000, random_state=SEED))
])
lr_default.fit(X_train, y_train)

# predict() applies threshold=0.5 internally: 1 if P>=0.5, else 0
y_pred_default = lr_default.predict(X_test)

# predict_proba() returns raw probabilities — we will tune the threshold ourselves
y_prob_default = lr_default.predict_proba(X_test)[:, 1]  # probability of spam

print("Default Logistic Regression (threshold = 0.5)")
print("=" * 50)
print(f"Accuracy    : {(y_pred_default == y_test).mean():.2%}")
print(f"Precision   : {precision_score(y_test, y_pred_default, zero_division=0):.2%}"
      "  (of flagged emails, % that are spam)")
print(f"Recall      : {recall_score(y_test, y_pred_default, zero_division=0):.2%}"
      "  (of all spam, % caught)")
print(f"F1 Score    : {f1_score(y_test, y_pred_default, zero_division=0):.4f}")
print()
print("Confusion matrix (rows=actual, cols=predicted):")
cm = confusion_matrix(y_test, y_pred_default)
print(pd.DataFrame(cm,
                    index=['Actual Not Spam', 'Actual Spam'],
                    columns=['Predicted Not Spam', 'Predicted Spam']))
print()
print(f"False Negatives (missed spam)    : {cm[1,0]}")
print(f"False Positives (legit as spam)  : {cm[0,1]}")
print()
print("OBSERVATION: Even with a trained model, low recall means most spam")
print("slips through because the model is calibrated for the majority class.")

## Section 4 — The Precision-Recall Curve

Instead of fixing threshold=0.5, we compute precision and recall **at every possible threshold** from 0 to 1. This gives the **PR curve** — the full picture of the precision-recall tradeoff.

**Average Precision (AP)** = area under the PR curve — the single best summary metric for imbalanced classification. Higher is better; a random classifier on 5% spam achieves AP ≈ 0.05.

In [ ]:
# ── Precision-Recall curve and optimal F1 threshold ───────────────────────────
# precision_recall_curve returns arrays sorted by decreasing threshold
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, y_prob_default)

# Compute F1 at every threshold (skip the last point which has no threshold)
# Avoid division by zero using np.where
f1_scores_thresh = np.where(
    (precisions[:-1] + recalls[:-1]) > 0,
    2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
    0.0
)

# Index of the threshold that gives the maximum F1
best_f1_idx = np.argmax(f1_scores_thresh)
best_f1_threshold = pr_thresholds[best_f1_idx]
best_f1_value     = f1_scores_thresh[best_f1_idx]

ap_score = average_precision_score(y_test, y_prob_default)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(recalls, precisions, lw=2, color=SPAM_COLOR, label=f'PR curve (AP={ap_score:.3f})')

# Mark the operating point at default threshold=0.5
prec_05 = precision_score(y_test, y_pred_default, zero_division=0)
rec_05  = recall_score(y_test, y_pred_default, zero_division=0)
ax.scatter([rec_05], [prec_05], s=150, color='orange', zorder=5,
            label=f'threshold=0.5  (P={prec_05:.2f}, R={rec_05:.2f})')

# Mark the optimal F1 threshold
ax.scatter([recalls[best_f1_idx]], [precisions[best_f1_idx]],
            s=150, marker='*', color=OPTIMAL_COLOR, zorder=5,
            label=f'Optimal F1 threshold={best_f1_threshold:.3f}\n'
                  f'(P={precisions[best_f1_idx]:.2f}, '
                  f'R={recalls[best_f1_idx]:.2f}, F1={best_f1_value:.3f})')

# Baseline AP for a random classifier equals the positive class fraction
ax.axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7,
            label=f'No-skill baseline (AP={y_test.mean():.2f})')

ax.set_xlabel('Recall (spam caught / total spam)', fontsize=11)
ax.set_ylabel('Precision (caught spam / flagged emails)', fontsize=11)
ax.set_title('Precision-Recall Curve\n'
              'Lower thresholds → upper-right (more recall, less precision)',
              fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Average Precision (AP) = {ap_score:.3f}")
print(f"Optimal F1 threshold   = {best_f1_threshold:.3f}")
print(f"Best F1 score          = {best_f1_value:.3f}")
print(f"Default (0.5) F1       = {f1_score(y_test, y_pred_default, zero_division=0):.3f}")

## Section 5 — ROC Curve and AUC

The **ROC curve** plots True Positive Rate (recall) vs False Positive Rate at every threshold. **AUC-ROC** = area under this curve.

**Important caveat for imbalanced data:**
- AUC-ROC can look impressive (e.g., 0.95) even when your model is failing the minority class.
- This is because FPR uses the *large* negative class as its denominator — small absolute numbers of FP get diluted.
- **PR curves are more informative when classes are imbalanced.** Always report both.

In [ ]:
# ── ROC curve and AUC ─────────────────────────────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob_default)
roc_auc = roc_auc_score(y_test, y_prob_default)

# Youden's J statistic: maximise (TPR - FPR) simultaneously
youden_j     = tpr - fpr
best_youden_idx = np.argmax(youden_j)
youden_threshold = roc_thresholds[best_youden_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ROC Curve vs PR Curve — Imbalanced Spam Data',
             fontsize=13, fontweight='bold')

# ── Left: ROC curve ────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(fpr, tpr, lw=2, color=SPAM_COLOR, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier (AUC=0.5)')
ax.scatter([fpr[best_youden_idx]], [tpr[best_youden_idx]],
            s=150, marker='*', color=OPTIMAL_COLOR, zorder=5,
            label=f"Youden's J max\n(threshold={youden_threshold:.3f})")
ax.set_xlabel('False Positive Rate (legit emails flagged / total legit)', fontsize=10)
ax.set_ylabel('True Positive Rate (spam caught / total spam)', fontsize=10)
ax.set_title('ROC Curve\nLooks impressive — but can mislead on imbalanced data', fontsize=10)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Right: PR curve (same model, different view) ───────────────────────────────
ax2 = axes[1]
ax2.plot(recalls, precisions, lw=2, color=NOSPAM_COLOR, label=f'PR curve (AP={ap_score:.3f})')
ax2.axhline(y=y_test.mean(), color='gray', linestyle='--', alpha=0.7,
             label=f'No-skill baseline (P={y_test.mean():.2f})')
ax2.scatter([recalls[best_f1_idx]], [precisions[best_f1_idx]],
             s=150, marker='*', color=OPTIMAL_COLOR, zorder=5,
             label=f'Best F1 (threshold={best_f1_threshold:.3f})')
ax2.set_xlabel('Recall (spam caught / total spam)', fontsize=10)
ax2.set_ylabel('Precision (caught spam / flagged emails)', fontsize=10)
ax2.set_title('PR Curve\nMore honest on imbalanced data — harder to look good here', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_vs_pr_curve.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"AUC-ROC = {roc_auc:.3f}  — sounds excellent")
print(f"AP (PR) = {ap_score:.3f}  — more honest about true model quality")
print(f"\nYouden's J optimal threshold: {youden_threshold:.3f}")
print(f"  TPR at Youden threshold    : {tpr[best_youden_idx]:.2%}")
print(f"  FPR at Youden threshold    : {fpr[best_youden_idx]:.2%}")

## Section 6 — Threshold Sweep: F1, Precision, Recall vs Threshold

This is the **central diagnostic plot** for threshold tuning. For every threshold from 0.05 to 0.95, we compute three metrics and plot them together. The optimal threshold is where the metric you care about is maximised.

In [ ]:
# ── Threshold sweep: precision, recall, F1 at every threshold ─────────────────
thresholds_sweep = np.linspace(0.05, 0.95, 91)  # 91 points from 0.05 to 0.95

prec_list, rec_list, f1_list = [], [], []

for thresh in thresholds_sweep:
    # Apply custom threshold: spam if P(spam) >= thresh
    y_pred_thresh = (y_prob_default >= thresh).astype(int)

    prec_list.append(precision_score(y_test, y_pred_thresh, zero_division=0))
    rec_list.append(recall_score(y_test, y_pred_thresh, zero_division=0))
    f1_list.append(f1_score(y_test, y_pred_thresh, zero_division=0))

prec_arr = np.array(prec_list)
rec_arr  = np.array(rec_list)
f1_arr   = np.array(f1_list)

best_sweep_f1_idx = np.argmax(f1_arr)
best_sweep_thresh = thresholds_sweep[best_sweep_f1_idx]

fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(thresholds_sweep, prec_arr, color=SPAM_COLOR,    lw=2, label='Precision')
ax.plot(thresholds_sweep, rec_arr,  color=NOSPAM_COLOR,  lw=2, label='Recall')
ax.plot(thresholds_sweep, f1_arr,   color=OPTIMAL_COLOR, lw=2.5, label='F1 Score')

# Vertical lines for key reference points
ax.axvline(x=0.5, color='orange', linestyle=':', lw=1.5, label='Default threshold (0.5)')
ax.axvline(x=best_sweep_thresh, color=OPTIMAL_COLOR, linestyle='--', lw=1.5,
            label=f'Optimal F1 threshold ({best_sweep_thresh:.2f})')

# Mark the F1 maximum with a dot
ax.scatter([best_sweep_thresh], [f1_arr[best_sweep_f1_idx]],
            s=120, color=OPTIMAL_COLOR, zorder=5)

ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Precision, Recall, and F1 Across All Thresholds\n'
              'As threshold increases: precision rises, recall falls',
              fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([-0.02, 1.05])
ax.grid(alpha=0.3)

# Annotate the crossing point of precision and recall
cross_idx = np.argmin(np.abs(prec_arr - rec_arr))
ax.annotate('P = R\n(breakeven)', xy=(thresholds_sweep[cross_idx], prec_arr[cross_idx]),
             xytext=(thresholds_sweep[cross_idx]+0.08, prec_arr[cross_idx]-0.1),
             arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

plt.tight_layout()
plt.savefig('threshold_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"At default threshold=0.5:")
print(f"  Precision={prec_arr[thresholds_sweep.tolist().index(min(thresholds_sweep, key=lambda x: abs(x-0.5)))]:.3f}, "
      f"  Recall={rec_arr[thresholds_sweep.tolist().index(min(thresholds_sweep, key=lambda x: abs(x-0.5)))]:.3f}")
print(f"At optimal F1 threshold={best_sweep_thresh:.2f}:")
print(f"  Precision={prec_arr[best_sweep_f1_idx]:.3f}, "
      f"  Recall={rec_arr[best_sweep_f1_idx]:.3f}, "
      f"  F1={f1_arr[best_sweep_f1_idx]:.3f}")

## Section 7 — Cost-Based Threshold Optimisation

F1 treats precision and recall as equally important. In the real world they rarely are. Here we define a **cost matrix** and find the threshold that minimises total expected cost:

$$\text{Total Cost} = C_{FP} \times N_{FP} + C_{FN} \times N_{FN}$$

For a spam filter:
- **FP cost = \$10**: A legitimate email gets deleted → user contacts support → customer service cost + risk of churn.
- **FN cost = \$0.01**: A spam email slips through → user is mildly annoyed → trivial cost.

This asymmetry tells us to use a **higher threshold** — we prefer letting spam through over blocking legitimate email.

In [ ]:
# ── Cost-based threshold optimisation ────────────────────────────────────────
# Cost per misclassification — defined by the business, not the data
COST_FP = 10.00   # cost of blocking a legitimate email ($)
COST_FN =  0.01   # cost of letting a spam email through ($)

total_costs = []

for thresh in thresholds_sweep:
    y_pred_thresh = (y_prob_default >= thresh).astype(int)
    cm_thresh = confusion_matrix(y_test, y_pred_thresh)

    # cm_thresh[actual, predicted]: [0,1] = FP (legit → spam), [1,0] = FN (spam → legit)
    n_fp = cm_thresh[0, 1] if cm_thresh.shape == (2, 2) else 0
    n_fn = cm_thresh[1, 0] if cm_thresh.shape == (2, 2) else 0

    cost = COST_FP * n_fp + COST_FN * n_fn
    total_costs.append(cost)

costs_arr = np.array(total_costs)
best_cost_idx   = np.argmin(costs_arr)
best_cost_thresh = thresholds_sweep[best_cost_idx]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(thresholds_sweep, costs_arr, color='darkorange', lw=2.5, label='Total cost ($)')
ax.axvline(x=best_cost_thresh, color=OPTIMAL_COLOR, linestyle='--', lw=2,
            label=f'Cost-optimal threshold = {best_cost_thresh:.2f}')
ax.axvline(x=0.5, color='gray', linestyle=':', lw=1.5, label='Default threshold (0.5)')
ax.axvline(x=best_sweep_thresh, color=SPAM_COLOR, linestyle=':', lw=1.5,
            label=f'F1-optimal threshold = {best_sweep_thresh:.2f}')

ax.scatter([best_cost_thresh], [costs_arr[best_cost_idx]],
            s=150, color=OPTIMAL_COLOR, zorder=5)
ax.annotate(f'Min cost = ${costs_arr[best_cost_idx]:,.2f}',
             xy=(best_cost_thresh, costs_arr[best_cost_idx]),
             xytext=(best_cost_thresh + 0.06, costs_arr[best_cost_idx] + 30),
             fontsize=10, color=OPTIMAL_COLOR,
             arrowprops=dict(arrowstyle='->', color=OPTIMAL_COLOR))

ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Total Expected Cost ($)', fontsize=12)
ax.set_title(f'Cost-Based Threshold Optimisation\n'
              f'(FP cost=${COST_FP:.2f}  |  FN cost=${COST_FN:.2f}  '
              f'→ FP is {COST_FP/COST_FN:.0f}× more costly than FN)',
              fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cost_threshold_optimization.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Cost matrix: FP=${COST_FP:.2f}  |  FN=${COST_FN:.2f}")
print(f"Cost-optimal threshold : {best_cost_thresh:.2f}")
print(f"Minimum total cost     : ${costs_arr[best_cost_idx]:,.2f}")
print(f"Cost at default (0.50) : ${costs_arr[np.argmin(np.abs(thresholds_sweep-0.5))]:,.2f}")
print(f"\nHigher cost of FP → higher optimal threshold → fewer (but more costly) false alarms")

## Section 8 — Class Weights: Addressing Imbalance During Training

**Threshold tuning is a post-training fix.** Class weighting addresses the imbalance **during** training by making each spam sample count more in the loss function.

Weighted cross-entropy loss:
$$L = -\frac{1}{n} \sum_{i=1}^{n} w_i \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

With `class_weight='balanced'`, sklearn sets:
$$w_\text{spam} = \frac{n_\text{total}}{n_\text{classes} \times n_\text{spam}} = \frac{10000}{2 \times 500} = 10$$

The optimiser now cares 10× more about getting spam right, so the decision boundary shifts toward the minority class.

In [ ]:
# ── Train with class_weight='balanced' and compare ────────────────────────────
lr_balanced = Pipeline([
    ("scaler", StandardScaler()),
    # 'balanced' automatically computes weights as n / (n_classes * class_count)
    ("lr", LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))
])
lr_balanced.fit(X_train, y_train)

y_pred_balanced = lr_balanced.predict(X_test)
y_prob_balanced = lr_balanced.predict_proba(X_test)[:, 1]

# ── Side-by-side confusion matrices ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Default vs Balanced Class Weights — Confusion Matrices',
             fontsize=13, fontweight='bold')

for ax, (y_pred, title) in zip(
    axes,
    [(y_pred_default,  'Default (class_weight=None)\nthreshold=0.5'),
     (y_pred_balanced, 'Balanced (class_weight="balanced")\nthreshold=0.5')]
):
    cm_plot = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm_plot, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Not Spam', 'Pred Spam'],
                yticklabels=['Actual Not Spam', 'Actual Spam'],
                linewidths=0.5, linecolor='gray')
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)
    ax.set_title(f"{title}\nP={p:.2f}  R={r:.2f}  F1={f:.3f}", fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrix_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("Key difference — balanced weights:")
print(f"  Recall improved from "
      f"{recall_score(y_test, y_pred_default, zero_division=0):.2%} to "
      f"{recall_score(y_test, y_pred_balanced, zero_division=0):.2%}")
print(f"  Precision dropped from "
      f"{precision_score(y_test, y_pred_default, zero_division=0):.2%} to "
      f"{precision_score(y_test, y_pred_balanced, zero_division=0):.2%}")
print()
print("This is expected: balanced weights push the boundary toward minority class,")
print("catching more spam but also flagging more legitimate email.")

In [ ]:
# ── Compare PR curves: default vs balanced ────────────────────────────────────
prec_bal, rec_bal, _ = precision_recall_curve(y_test, y_prob_balanced)
ap_balanced = average_precision_score(y_test, y_prob_balanced)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(recalls,  precisions, lw=2, color=SPAM_COLOR,
         label=f'Default (AP={ap_score:.3f})')
ax.plot(rec_bal,  prec_bal,   lw=2, color=NOSPAM_COLOR, linestyle='--',
         label=f'Balanced weights (AP={ap_balanced:.3f})')
ax.axhline(y=y_test.mean(), color='gray', linestyle=':', alpha=0.7,
            label=f'No-skill baseline (P={y_test.mean():.2f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('PR Curve: Default vs Balanced Class Weights\n'
              'Higher AP = better model for the minority class', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.05])

plt.tight_layout()
plt.savefig('pr_curve_balanced_vs_default.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Default model AP    : {ap_score:.4f}")
print(f"Balanced model AP   : {ap_balanced:.4f}")
print(f"AP improvement      : +{(ap_balanced - ap_score):.4f}")
print()
print("A higher AP curve means that for any given recall level,")
print("the balanced model achieves better precision.")

## Section 9 — F-beta Scores: Formalising the Precision-Recall Preference

We now compute F-beta scores at optimal thresholds for each beta value and compare them in a structured table.

| Beta | Model use case | Optimal threshold behaviour |
|---|---|---|
| 0.5 | Spam filter (annoying false alarms) | Higher threshold → conservative |
| 1.0 | Balanced: no strong preference | Middle ground |
| 2.0 | Cancer screening (missing case is catastrophic) | Lower threshold → aggressive |


In [ ]:
# ── F-beta scores at threshold optimised for each beta ────────────────────────
betas = [0.5, 1.0, 2.0]
results_table = []

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('F-beta Score vs Threshold for β ∈ {0.5, 1.0, 2.0}\n'
             'Each beta has a different optimal threshold',
             fontsize=13, fontweight='bold')

colors_beta = ['#9467bd', OPTIMAL_COLOR, '#17becf']

for ax, beta, clr in zip(axes, betas, colors_beta):
    fbeta_list = []

    for thresh in thresholds_sweep:
        y_pred_t = (y_prob_default >= thresh).astype(int)
        # fbeta_score: higher beta means recall is weighted more heavily
        fb = fbeta_score(y_test, y_pred_t, beta=beta, zero_division=0)
        fbeta_list.append(fb)

    fbeta_arr = np.array(fbeta_list)
    best_beta_idx = np.argmax(fbeta_arr)
    best_beta_thresh = thresholds_sweep[best_beta_idx]

    ax.plot(thresholds_sweep, fbeta_arr, lw=2, color=clr, label=f'F{beta} score')
    ax.axvline(x=best_beta_thresh, color='black', linestyle='--', lw=1.5,
                label=f'Optimal threshold = {best_beta_thresh:.2f}')
    ax.scatter([best_beta_thresh], [fbeta_arr[best_beta_idx]],
                s=120, color='black', zorder=5)
    ax.axvline(x=0.5, color='gray', linestyle=':', lw=1, label='Default=0.5')

    use_case = {0.5: 'Spam filter\n(precision priority)',
                1.0: 'Balanced use case',
                2.0: 'Medical test\n(recall priority)'}[beta]
    ax.set_title(f'β = {beta}  |  {use_case}\nMax F{beta} = {fbeta_arr[best_beta_idx]:.3f}',
                  fontsize=10, fontweight='bold')
    ax.set_xlabel('Threshold', fontsize=10)
    ax.set_ylabel(f'F{beta} Score', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim([-0.02, 0.85])

    # Gather results for summary table
    y_pred_opt = (y_prob_default >= best_beta_thresh).astype(int)
    results_table.append({
        'Beta': beta,
        'Use Case': use_case.replace('\n', ' '),
        'Optimal Threshold': round(best_beta_thresh, 3),
        'Precision': round(precision_score(y_test, y_pred_opt, zero_division=0), 3),
        'Recall': round(recall_score(y_test, y_pred_opt, zero_division=0), 3),
        f'F{beta}': round(fbeta_arr[best_beta_idx], 3)
    })

plt.tight_layout()
plt.savefig('fbeta_curves.png', dpi=120, bbox_inches='tight')
plt.show()

# Print summary table
print("\nSummary: Optimal Threshold per Beta")
print("=" * 80)
df_results = pd.DataFrame(results_table)
print(df_results.to_string(index=False))
print()
print("INSIGHT: Higher beta → lower optimal threshold (more aggressive spam catching)")
print("Lower beta → higher optimal threshold (fewer false alarms, even if spam slips through)")

## Section 10 — Complete Strategy Comparison

Bringing it all together: we compare every strategy in one table.

In [ ]:
# ── Comprehensive strategy comparison table ───────────────────────────────────
strategies = [
    ("Always predict Not-Spam",           y_pred_baseline),
    ("Default LR (threshold=0.50)",        y_pred_default),
    ("LR with balanced weights (t=0.50)",  y_pred_balanced),
    ("Default LR (F1-optimal threshold)",
     (y_prob_default >= best_sweep_thresh).astype(int)),
    ("Default LR (cost-optimal threshold)",
     (y_prob_default >= best_cost_thresh).astype(int)),
    ("Default LR (Youden's J threshold)",
     (y_prob_default >= youden_threshold).astype(int)),
]

rows = []
for name, y_pred in strategies:
    rows.append({
        'Strategy': name,
        'Accuracy': f"{(y_pred == y_test).mean():.2%}",
        'Spam Precision': f"{precision_score(y_test, y_pred, zero_division=0):.2%}",
        'Spam Recall': f"{recall_score(y_test, y_pred, zero_division=0):.2%}",
        'F1 (spam)': f"{f1_score(y_test, y_pred, zero_division=0):.4f}",
        'Spam caught': f"{int(y_pred[y_test==1].sum())}/{int(y_test.sum())}",
        'FP (legit blocked)': f"{int(((y_pred==1) & (y_test==0)).sum())}"
    })

df_summary = pd.DataFrame(rows)
print("Strategy Comparison (Test Set)")
print("=" * 120)
print(df_summary.to_string(index=False))
print()
print("KEY TAKEAWAYS:")
print("  1. Accuracy alone is a misleading metric on imbalanced data.")
print("  2. Threshold tuning and class weights address different parts of the pipeline.")
print("  3. The 'best' strategy depends entirely on the cost of FP vs FN.")
print("  4. Always define your evaluation metric BEFORE training — not after.")

## Self-Check Questions

Work through these before moving on. The answers require you to reason about concepts, not just look up a number.

---

**Q1.** Your spam filter has precision = 0.95, recall = 0.40 at threshold = 0.5. What does this mean for users? Should you lower or raise the threshold?

<details>
<summary>Show answer</summary>

**Interpretation:**
- Precision = 0.95: Of every 100 emails flagged as spam, 95 are actually spam. Very few false alarms — legitimate emails are rarely blocked. Good for user experience.
- Recall = 0.40: Of every 100 real spam emails, only 40 are caught. A full 60% of spam slips through to the inbox. Users are frequently annoyed by spam.

**For users:** The filter is conservative. It only flags emails it is very sure about, so users rarely miss a legitimate email — but they see a lot of spam.

**Action:** Lower the threshold (e.g., to 0.20–0.30). This will:
- Increase recall — more spam is caught
- Decrease precision — some legitimate emails may be blocked

Whether the tradeoff is acceptable depends on your cost matrix. If blocking legitimate email (FP) is more costly than seeing spam (FN), keep the high threshold. If user annoyance from spam is more costly, lower it.

</details>

---

**Q2.** AUC-ROC = 0.98 for a model on a 99:1 imbalanced dataset. Is this impressive? What additional metric would you check?

<details>
<summary>Show answer</summary>

**Not necessarily impressive — you must check the PR curve (Average Precision).**

On a 99:1 dataset, the False Positive Rate (FPR = FP / N) uses the large negative class as denominator. Even if your model makes many false positives in absolute terms, they are diluted by the huge denominator, keeping FPR low and ROC AUC high.

Example: A model that always predicts class 1 has perfect recall (1.0) and FPR = 1.0 → AUC = 0.5. But a model that gets 90% of class 1 right while making 5% of class 0 wrong will still look great on ROC because 5% of 99% ≈ 4.95% FPR, while TPR = 90%.

**Check next:** Average Precision (area under PR curve). On a 99:1 dataset, a random classifier has AP ≈ 0.01. If your AP is 0.30, that is 30× better than random — which is truly impressive. If AP is 0.95, the model is genuinely excellent.

Always report both AUC-ROC and AP on imbalanced datasets. AUC-ROC alone can mask model deficiencies on the minority class.

</details>

---

**Q3.** Your bank sets: misclassifying a fraudulent transaction (FN) costs \$500 and a false alarm (FP) costs \$5. What threshold optimisation approach would you use?

<details>
<summary>Show answer</summary>

**Use cost-based threshold optimisation** with the given cost matrix:

$$\text{Total Cost}(t) = 500 \times N_{FN}(t) + 5 \times N_{FP}(t)$$

Sweep the threshold from 0 to 1, compute this cost at each threshold using the validation set, and select the threshold that minimises total cost.

**Intuition:** FN is 100× more costly than FP. The cost function will strongly penalise missing fraud, pushing the optimal threshold very low (perhaps 0.10–0.15). At that threshold, the model flags many legitimate transactions as suspicious (high FP count), but at only $5 each, the total FP cost is manageable. Missing even one fraud at $500 overwhelms hundreds of false alarms.

**Alternative formulation:** The optimal threshold in theory is:
$$t^* = \frac{C_{FP}}{C_{FP} + C_{FN}} = \frac{5}{5 + 500} = 0.0099$$

This analytical result holds when the model is well-calibrated. In practice, sweep empirically on held-out validation data.

</details>

---

**Q4.** How does `class_weight='balanced'` in sklearn change the training process? Does it change the model architecture or just the loss function?

<details>
<summary>Show answer</summary>

**It changes only the loss function — the model architecture is identical.**

`class_weight='balanced'` multiplies each training sample's contribution to the loss by a weight inversely proportional to its class frequency:

$$w_c = \frac{n_{\text{total}}}{n_{\text{classes}} \times n_c}$$

For our 95:5 split: $w_\text{spam} = \frac{10000}{2 \times 500} = 10$, $w_\text{legit} = \frac{10000}{2 \times 9500} \approx 0.53$

**Effect:** The gradient from each spam sample is 10× larger than from a legitimate sample. The optimiser must work much harder to reduce the loss on spam emails, so the final weights (and hence the decision boundary) shift toward the spam class.

**What does NOT change:**
- Number of parameters
- Model form (still logistic regression with a linear boundary)
- Prediction function (still σ(**w**ᵀ**x** + b))

**What DOES change:**
- The position and orientation of the decision boundary (it shifts toward the minority class)
- The effective threshold: `class_weight='balanced'` at threshold=0.5 is roughly equivalent to lowering the threshold on the default model

**Important:** After applying class weights, the model's `predict_proba()` output is no longer well-calibrated — the probabilities no longer accurately reflect empirical frequencies. If you need calibrated probabilities, apply Platt scaling or isotonic regression on top.

</details>